In [ ]:
from dotenv import load_dotenv
load_dotenv()

import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt

llm = init_chat_model("openai:gpt-4o")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config={
    "configurable": {
        "thread_id": "1",
    },
    "recursion_limit": 2,
}

In [2]:
class State(MessagesState):
    custom_stuff: str

graph_builder = StateGraph(State)

In [3]:
@tool
def get_human_feedback(poem: str):
    """ Asks the user for feedback on. the poem.
        Use this before returning the final response.
    """
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback


llm_with_tools = llm.bind_tools(tools=[get_human_feedback]) # LLM에게 Tool을 설명해줌.

def chatbot(state: State):
    response = llm_with_tools.invoke(f"""
        You are an expert in making poems.
        Use the 'get_human_feedback' tool to get feedback on your poem.
        Only after you receive positive feedback you can return the final poem.
        ALWAYS ASK FOR FEEDBACK FIRST.
        Here is the conversation history:
        {state["messages"]}
    """)
    return { "messages": [response] }
    

In [4]:
tool_node = ToolNode(
    tools=[
        get_human_feedback,
    ],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [5]:
result = graph.invoke(
    {
        "messages": [
            { "role": "user", "content": "Please make a poem about Python code." }
        ]
    },
    config= config
)

In [6]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================

In the realm of logic and light,  
Where algorithms dance in delight,  
Python weaves a tale so grand,  
A language both humble and in command.  

With serpentine grace, it winds through lines,  
Transforming code into elegant designs,  
Indentations as its guiding star,  
Each block a testament to how neat things are.  

No need for semicolons to restrain,  
Just clear syntax, like gentle rain,  
Flowing smoothly, a river's path,  
Avoiding complexity's aftermath.  

Libraries vast, a treasure trove,  
NumPy, Pandas, they deftly rove,  
Matplotlib paints with colors so bright,  
Creating vistas in digital light.  

Through loops and conditions, it deftly turns,  
In its cauldron of possibilities, it churns,  
Crafting solutions with ease and grace,  
In Python's embrace, errors we erase

In [8]:
snapshot = graph.get_state(config)

# snapshot.interrupts
snapshot.next

('tools',)

In [10]:
from langgraph.types import Command

response = Command(
    # 아래와 같이 하려면 feedback 반환값을 feedback["feedback"] 으로 수정해야함.
    # resume={ "feedback": "It looks good!" }
    resume="It looks good!"
)

result = graph.invoke(
    response,
    config= config,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================

In the realm of logic and light,  
Where algorithms dance in delight,  
Python weaves a tale so grand,  
A language both humble and in command.  

With serpentine grace, it winds through lines,  
Transforming code into elegant designs,  
Indentations as its guiding star,  
Each block a testament to how neat things are.  

No need for semicolons to restrain,  
Just clear syntax, like gentle rain,  
Flowing smoothly, a river's path,  
Avoiding complexity's aftermath.  

Libraries vast, a treasure trove,  
NumPy, Pandas, they deftly rove,  
Matplotlib paints with colors so bright,  
Creating vistas in digital light.  

Through loops and conditions, it deftly turns,  
In its cauldron of possibilities, it churns,  
Crafting solutions with ease and grace,  
In Python's embrace, errors we erase

In [ ]:
snapshot = graph.get_state(config)

# snapshot.interrupts
snapshot.next # 종료되면 '()'로 출력됨.

()